In [1]:
pip install boxmot motmetrics

  Using cached scikit_learn-1.7.2-cp310-cp310-win_amd64.whl.metadata (11 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached PySocks-1.7.1-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   --------------------- ------------------ 1.3/2.4 MB 7.4 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 6

In [3]:
import cv2
import time
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm import tqdm

from ultralytics import YOLO

import motmetrics as mm

In [38]:
from ultralytics import YOLO


# -------------------------------
# YOLO11s COCO pretrained baseline
# -------------------------------
model_base = YOLO(
    "yolo11s.pt"
)

model_base.fuse()



# -------------------------------
# YOLO11s VisDrone fine-tuned
# -------------------------------
model_ft = YOLO(
    r"C:\Users\VAMSEEKRISHNA.P\Desktop\assignment\runs\detect\runs\YOLO11s_VisDrone_1280\weights\best.pt"
)

model_ft.fuse()

YOLO11s summary (fused): 100 layers, 9,443,760 parameters, 0 gradients, 21.5 GFLOPs
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs


YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(96, 128, kernel_size=(1, 1), stride=(1, 1))
          (act): SiLU(inplace=True)
        )
        (m): ModuleList(
          (0): Bottleneck(
            (cv1): Conv(
              (conv): Conv2d(32, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
              (act): SiLU(inplace=True)
            )
            (cv2): Conv(
              (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
              (act): SiLU(inplace=True)
 

In [39]:
VAL_ROOT = Path(
    r"C:\Users\VAMSEEKRISHNA.P\Desktop\assignment\VisDrone2019-MOT-val\VisDrone2019-MOT-val"
)

SEQ_DIR = VAL_ROOT / "sequences"

ANN_DIR = VAL_ROOT / "annotations"


sequences = sorted(
    list(SEQ_DIR.iterdir())
)[:2]


print(
    "Testing sequences:"
)

for s in sequences:
    print(s.name)

Testing sequences:
uav0000086_00000_v
uav0000117_02622_v


In [26]:
def load_gt(seq_name):

    file = ANN_PATH / (seq_name+".txt")


    gt={}


    with open(file,"r") as f:


        for line in f:

            d=line.strip().split(",")


            frame=int(d[0])

            tid=int(d[1])


            x=float(d[2])
            y=float(d[3])
            w=float(d[4])
            h=float(d[5])


            cls=int(d[7])


            # persons only
            if cls not in [1,2]:
                continue


            if frame not in gt:
                gt[frame]=[]



            gt[frame].append(
                [
                    tid,
                    x,
                    y,
                    w,
                    h
                ]
            )


    return gt

In [27]:
def calc_iou(
    gt,
    pred
):


    gt_boxes=[
        g[1:]
        for g in gt
    ]

    pred_boxes=[
        p[1:]
        for p in pred
    ]


    return mm.distances.iou_matrix(

        np.asarray(
            gt_boxes,
            dtype=float
        ),

        np.asarray(
            pred_boxes,
            dtype=float
        ),

        max_iou=0.5
    )

In [43]:
def evaluate_tracker(detector, tracker_yaml):


    acc = mm.MOTAccumulator(
        auto_id=True
    )


    total_frames=0

    start=time.time()



    for seq in sequences:


        print(
            "Sequence:",
            seq.name
        )


        gt = load_gt(
            seq.name
        )


        frames = sorted(
            seq.glob("*.jpg")
        )


        # reset tracker memory
        detector.predictor=None



        for idx,img_path in enumerate(
            tqdm(frames)
        ):


            frame_id=idx+1


            frame=cv2.imread(
                str(img_path)
            )



            result= detector.track(

                frame,

                imgsz=1280,

                conf=0.05,

                classes=[0],

                tracker=tracker_yaml,

                persist=True,

                device=0,

                verbose=False

            )[0]



            preds=[]


            if result.boxes.id is not None:


                ids=result.boxes.id.cpu().numpy()

                boxes=result.boxes.xywh.cpu().numpy()



                for tid,box in zip(
                    ids,
                    boxes
                ):


                    x,y,w,h=box


                    preds.append(

                        [
                            int(tid),

                            x-w/2,

                            y-h/2,

                            w,

                            h
                        ]

                    )


            gt_frame = gt.get(
                frame_id,
                []
            )



            acc.update(

                [g[0] for g in gt_frame],

                [p[0] for p in preds],

                calc_iou(
                    gt_frame,
                    preds
                )

            )


            total_frames+=1



    fps = (
        total_frames /
        (time.time()-start)
    )


    return acc,fps

In [47]:
experiments = [

    {
        "name": "YOLO11s_Base_ByteTrack",
        "model": model_base,
        "tracker": r"Trackers\bytetrack_drone.yaml"
    },

    {
        "name": "YOLO11s_Base_BoTSORT_GMC",
        "model": model_base,
        "tracker": r"Trackers\botsort_gmc_drone.yaml"
    },


    {
        "name": "YOLO11s_FT_ByteTrack",
        "model": model_ft,
        "tracker": r"Trackers\bytetrack_drone.yaml"
    },

    {
        "name": "YOLO11s_FT_BoTSORT_GMC",
        "model": model_ft,
        "tracker": r"Trackers\botsort_gmc_drone.yaml"
    }

]

In [48]:
import cv2
import time
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm import tqdm

from ultralytics import YOLO

import motmetrics as mm


# -------------------------
# NumPy 2.0 patch
# -------------------------
if not hasattr(np, "asfarray"):
    np.asfarray = lambda x: np.asarray(x, dtype=float)

In [49]:
results=[]

mh=mm.metrics.create()


for exp in experiments:


    print(
        "\nRunning:",
        exp["name"]
    )


    acc,fps = evaluate_tracker(
        exp["model"],
        exp["tracker"]
    )


    summary = mh.compute(
        acc,
        metrics=[
            "mota",
            "idf1",
            "num_switches",
            "mostly_tracked",
            "mostly_lost",
            "precision",
            "recall"
        ],
        name="result"
    )


    results.append({

        "Pipeline":exp["name"],

        "FPS":fps,

        "MOTA":summary["mota"].iloc[0],

        "IDF1":summary["idf1"].iloc[0],

        "ID Switches":summary["num_switches"].iloc[0],

        "Mostly Tracked":summary["mostly_tracked"].iloc[0],

        "Mostly Lost":summary["mostly_lost"].iloc[0],

        "Precision":summary["precision"].iloc[0],

        "Recall":summary["recall"].iloc[0]

    })


df=pd.DataFrame(results)

df


Running: YOLO11s_Base_ByteTrack
Sequence: uav0000086_00000_v


100%|██████████| 464/464 [00:14<00:00, 31.35it/s]


Sequence: uav0000117_02622_v


100%|██████████| 349/349 [00:15<00:00, 22.65it/s]



Running: YOLO11s_Base_BoTSORT_GMC
Sequence: uav0000086_00000_v


...


Running: YOLO11s_FT_BoTSORT_GMC
Sequence: uav0000086_00000_v


100%|██████████| 464/464 [00:47<00:00,  9.78it/s]


Sequence: uav0000117_02622_v


100%|██████████| 349/349 [01:26<00:00,  4.04it/s]


,Pipeline,FPS,MOTA,IDF1,ID Switches,Mostly Tracked,Mostly Lost,Precision,Recall
0,YOLO11s_Base_ByteTrack,26.844614,0.198281,0.324777,38,6,102,0.870368,0.234387
1,YOLO11s_Base_BoTSORT_GMC,5.332247,0.176435,0.299997,8,7,101,0.862644,0.210149
2,YOLO11s_FT_ByteTrack,26.117115,0.405502,0.517980,228,23,56,0.787576,0.565097
3,YOLO11s_FT_BoTSORT_GMC,6.073024,0.287679,0.440554,27,17,84,0.837382,0.358065


In [50]:
results_df = pd.DataFrame(results)
results_df

,Pipeline,FPS,MOTA,IDF1,ID Switches,Mostly Tracked,Mostly Lost,Precision,Recall
0,YOLO11s_Base_ByteTrack,26.844614,0.198281,0.324777,38,6,102,0.870368,0.234387
1,YOLO11s_Base_BoTSORT_GMC,5.332247,0.176435,0.299997,8,7,101,0.862644,0.210149
2,YOLO11s_FT_ByteTrack,26.117115,0.405502,0.517980,228,23,56,0.787576,0.565097
3,YOLO11s_FT_BoTSORT_GMC,6.073024,0.287679,0.440554,27,17,84,0.837382,0.358065


In [51]:
from ultralytics import YOLO
import cv2
import time
import numpy as np
from pathlib import Path
from collections import defaultdict


# base COCO model
model_base = YOLO(
    "yolo11s.pt"
)


# fine tuned model
model_ft = YOLO(
    r"C:\Users\VAMSEEKRISHNA.P\Desktop\assignment\runs\detect\runs\YOLO11s_VisDrone_1280\weights\best.pt"
)


model_base.fuse()
model_ft.fuse()

YOLO11s summary (fused): 100 layers, 9,443,760 parameters, 0 gradients, 21.5 GFLOPs
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs


YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(96, 128, kernel_size=(1, 1), stride=(1, 1))
          (act): SiLU(inplace=True)
        )
        (m): ModuleList(
          (0): Bottleneck(
            (cv1): Conv(
              (conv): Conv2d(32, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
              (act): SiLU(inplace=True)
            )
            (cv2): Conv(
              (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
              (act): SiLU(inplace=True)
 

In [53]:
SEQ = Path(
    r"C:\Users\VAMSEEKRISHNA.P\Desktop\assignment\VisDrone2019-MOT-val\VisDrone2019-MOT-val\sequences\uav0000086_00000_v"
)


frames = sorted(
    SEQ.glob("*.jpg")
)


print(
    "Frames:",
    len(frames)
)

Frames: 464


In [56]:
def generate_tracking_video(
        model,
        output_name
):


    first = cv2.imread(
        str(frames[0])
    )


    h,w,_ = first.shape



    writer = cv2.VideoWriter(

        output_name,

        cv2.VideoWriter_fourcc(
            *"mp4v"
        ),

        30,

        (w,h)
    )



    trajectories = defaultdict(list)



    # reset tracker memory
    model.predictor = None



    total_time = 0



    for img_path in frames:



        frame = cv2.imread(
            str(img_path)
        )



        start = time.time()



        result = model.track(

            frame,

            imgsz=1280,

            conf=0.05,

            iou=0.5,

            classes=[0],

            tracker=r"Trackers\bytetrack_drone.yaml",

            persist=True,

            device=0,

            verbose=False

        )[0]



        end = time.time()


        fps = 1/(end-start)

        total_time += end-start




        if result.boxes.id is not None:



            ids = (
                result.boxes.id
                .cpu()
                .numpy()
            )


            boxes = (
                result.boxes.xyxy
                .cpu()
                .numpy()
            )




            for tid,box in zip(
                ids,
                boxes
            ):


                x1,y1,x2,y2 = map(
                    int,
                    box
                )



                cx = int(
                    (x1+x2)/2
                )

                cy = int(
                    (y1+y2)/2
                )



                trajectories[int(tid)].append(
                    (cx,cy)
                )


                # keep last 30 points
                trajectories[int(tid)] = (
                    trajectories[int(tid)][-30:]
                )




                # bbox
                cv2.rectangle(

                    frame,

                    (x1,y1),

                    (x2,y2),

                    (0,255,0),

                    2

                )



                # ID
                cv2.putText(

                    frame,

                    f"ID:{int(tid)}",

                    (x1,y1-5),

                    cv2.FONT_HERSHEY_SIMPLEX,

                    0.5,

                    (0,255,0),

                    2

                )



                # trajectory

                pts = np.array(
                    trajectories[int(tid)],
                    np.int32
                )


                if len(pts)>1:

                    cv2.polylines(

                        frame,

                        [pts],

                        False,

                        (0,0,255),

                        2

                    )




        # FPS display
        cv2.putText(

            frame,

            f"FPS:{fps:.1f}",

            (30,40),

            cv2.FONT_HERSHEY_SIMPLEX,

            1,

            (255,0,0),

            2

        )



        writer.write(
            frame
        )



    writer.release()



    print(
        output_name,
        "saved"
    )


    print(
        "Average FPS:",
        len(frames)/total_time
    )

In [57]:
generate_tracking_video(

    model_base,

    "YOLO11s_Base_ByteTrack.mp4"

)


generate_tracking_video(

    model_ft,

    "YOLO11s_FT_ByteTrack.mp4"

)

YOLO11s_Base_ByteTrack.mp4 saved
Average FPS: 32.35999716598519
YOLO11s_FT_ByteTrack.mp4 saved
Average FPS: 24.990883827591766


In [63]:
def generate_tracking_video(
        model,
        output_name
):


    first = cv2.imread(
        str(frames[0])
    )


    h,w,_ = first.shape



    writer = cv2.VideoWriter(

        output_name,

        cv2.VideoWriter_fourcc(
            *"avc1"
        ),

        30,

        (w,h)
    )



    trajectories = defaultdict(list)



    # reset tracker memory
    model.predictor = None



    total_time = 0



    for img_path in frames:



        frame = cv2.imread(
            str(img_path)
        )



        start = time.time()



        result = model.track(

            frame,

            imgsz=1280,

            conf=0.01,

            iou=0.5,

            max_det=1000,


            classes=[0],

            tracker=r"Trackers\bytetrack_drone.yaml",

            persist=True,

            device=0,

            verbose=False

        )[0]



        end = time.time()


        fps = 1/(end-start)

        total_time += end-start




        if result.boxes.id is not None:



            ids = (
                result.boxes.id
                .cpu()
                .numpy()
            )


            boxes = (
                result.boxes.xyxy
                .cpu()
                .numpy()
            )




            for tid,box in zip(
                ids,
                boxes
            ):


                x1,y1,x2,y2 = map(
                    int,
                    box
                )



                cx = int(
                    (x1+x2)/2
                )

                cy = int(
                    (y1+y2)/2
                )



                trajectories[int(tid)].append(
                    (cx,cy)
                )


                # keep last 30 points
                trajectories[int(tid)] = (
                    trajectories[int(tid)][-30:]
                )




                # bbox
                cv2.rectangle(

                    frame,

                    (x1,y1),

                    (x2,y2),

                    (0,255,0),

                    2

                )



                # ID
                cv2.putText(

                    frame,

                    f"ID:{int(tid)}",

                    (x1,y1-5),

                    cv2.FONT_HERSHEY_SIMPLEX,

                    0.5,

                    (0,255,0),

                    2

                )



                # trajectory

                pts = np.array(
                    trajectories[int(tid)],
                    np.int32
                )


                if len(pts)>1:

                    cv2.polylines(

                        frame,

                        [pts],

                        False,

                        (0,0,255),

                        2

                    )




        # FPS display
        cv2.putText(

            frame,

            f"FPS:{fps:.1f}",

            (30,40),

            cv2.FONT_HERSHEY_SIMPLEX,

            1,

            (255,0,0),

            2

        )



        writer.write(
            frame
        )



    writer.release()



    print(
        output_name,
        "saved"
    )


    print(
        "Average FPS:",
        len(frames)/total_time
    )

In [64]:
generate_tracking_video(

    model_base,

    "YOLO11s_Base_ByteTrack_avc1.mp4"

)


generate_tracking_video(

    model_ft,

    "YOLO11s_FT_ByteTrack_avc1.mp4"

)

YOLO11s_Base_ByteTrack_avc1.mp4 saved
Average FPS: 6.391565863260331
YOLO11s_FT_ByteTrack_avc1.mp4 saved
Average FPS: 5.678780631105903
